In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime

In [0]:
df_fact_silver = spark.table("vg_sales.02_silver.fact_vg_sales_main")
display(df_fact_silver)

In [0]:
# Lendo da Fato Silver
df_fato = spark.table("vg_sales.02_silver.fact_vg_sales_main")

# Criando o DataFrame manual para Console
df_dim_console = df_fato.select(
    F.sha2(F.lower(F.trim(F.col("console"))), 256).alias("console_hash"),
    F.regexp_replace(F.trim(F.col("console")), " +", " ").alias("console_name"),
    F.current_timestamp().alias("start_date"),
    F.lit(None).cast("timestamp").alias("end_date"),
    F.lit(True).alias("is_current")
).distinct().filter("console_name IS NOT NULL")

# Registrando para o SQL
df_dim_console.createOrReplaceTempView("src_console")

# Merge Manual
spark.sql("""
    MERGE INTO vg_sales.02_silver.dim_console AS target
    USING src_console AS source
    ON target.console_hash = source.console_hash
    WHEN NOT MATCHED THEN
      INSERT (console_hash, console_name, start_date, end_date, is_current)
      VALUES (source.console_hash, source.console_name, source.start_date, source.end_date, source.is_current)
""")

In [0]:
from pyspark.sql import functions as F

# Lendo da Fato Silver
df_fato = spark.table("vg_sales.02_silver.fact_vg_sales_main")

# Criando o DataFrame manual para Genre
df_dim_genre = df_fato.select(
    # 1. Padroniza o nome (trim + remove espaços duplos)
    F.regexp_replace(F.trim(F.col("genre")), " +", " ").alias("genre_name")
).select(
    # 2. Gera o Hash baseado no nome limpo e minúsculo
    F.sha2(F.lower(F.col("genre_name")), 256).alias("genre_hash"),
    F.col("genre_name"),
    F.current_timestamp().alias("start_date"),
    F.lit(None).cast("timestamp").alias("end_date"),
    F.lit(True).alias("is_current")
).distinct().filter("genre_name IS NOT NULL")

# Registrando para o SQL
df_dim_genre.createOrReplaceTempView("src_genre")

# Merge Manual
spark.sql("""
    MERGE INTO vg_sales.02_silver.dim_genre AS target
    USING src_genre AS source
    ON target.genre_hash = source.genre_hash
    WHEN NOT MATCHED THEN
      INSERT (genre_hash, genre_name, start_date, end_date, is_current)
      VALUES (source.genre_hash, source.genre_name, source.start_date, source.end_date, source.is_current)
""")

In [0]:
# Criando o DataFrame manual para Publisher
df_dim_publisher = df_fato.select(
    # 1. Padroniza o nome (trim + remove espaços duplos)
    F.regexp_replace(F.trim(F.col("publisher")), " +", " ").alias("publisher_name")
).select(
    # 2. Gera o Hash baseado no nome limpo e minúsculo
    F.sha2(F.lower(F.col("publisher_name")), 256).alias("publisher_hash"),
    F.col("publisher_name"),
    F.current_timestamp().alias("start_date"),
    F.lit(None).cast("timestamp").alias("end_date"),
    F.lit(True).alias("is_current")
).distinct().filter("publisher_name IS NOT NULL")

# Registrando para o SQL
df_dim_publisher.createOrReplaceTempView("src_publisher")

# Merge Manual
spark.sql("""
    MERGE INTO vg_sales.02_silver.dim_publisher AS target
    USING src_publisher AS source
    ON target.publisher_hash = source.publisher_hash
    WHEN NOT MATCHED THEN
      INSERT (publisher_hash, publisher_name, start_date, end_date, is_current)
      VALUES (source.publisher_hash, source.publisher_name, source.start_date, source.end_date, source.is_current)
""")

In [0]:
%sql
--select * from vg_sales.02_silver.dim_console
--select * from vg_sales.02_silver.dim_publisher
select * from vg_sales.02_silver.dim_genre